In [ ]:
!pip install -q pywavelets scikit-image evaluate

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import os, sys, gc, json, random, warnings
from io import BytesIO
from pathlib import Path
from typing import Dict
from collections import Counter

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import Dataset, Image as HFImage, ClassLabel
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from torchvision.transforms import (
    Compose, ToTensor, Normalize,
    RandomHorizontalFlip, RandomRotation, RandomResizedCrop
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, roc_auc_score, classification_report
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import pywt
from scipy import stats
from scipy.ndimage import uniform_filter, gaussian_filter
from scipy.fftpack import dct as scipy_dct

ImageFile.LOAD_TRUNCATED_IMAGES = True

# GPU check
if not torch.cuda.is_available():
    print('No GPU detected!'); sys.exit(1)
NUM_GPUS = torch.cuda.device_count()
DEVICE = 'cuda'
for i in range(NUM_GPUS):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}  '
          f'{torch.cuda.get_device_properties(i).total_memory/1024**3:.1f} GB')
print('Imports OK')

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Dataset paths
AI_FOLDER   = '/kaggle/input/datasets/arunmass/ai-image-vs-real-image/ai/ai'
REAL_FOLDER = '/kaggle/input/datasets/arunmass/ai-image-vs-real-image/real/real'
OUTPUT_DIR  = '/kaggle/working/hybrid_forensic_512'

# Phase 1 features.csv — pre-computed, load directly
FEATURES_CSV = '/kaggle/input/datasets/arunmass/feature-ai-real/features.csv'

# Model — fine-tuned SigLIP already trained on AI vs Human
MODEL_ID = 'Ateeqq/ai-vs-human-image-detector'

# Resolution — must match Phase 1 feature extraction (512)
INPUT_RESOLUTION = 512

# Split: 70% train | 15% val | 15% test
VAL_SIZE  = 0.15
TEST_SIZE = 0.15

# Training hyperparams
MAX_EPOCHS          = 25
EARLY_STOP_PATIENCE = 5
BATCH_SIZE_PER_GPU  = 8
GRAD_ACCUM_STEPS    = 4      # effective BS = 8 * GPUs * 4
LEARNING_RATE       = 2e-6   # backbone (already fine-tuned, use very low LR)
HEAD_LR             = 3e-4   # hybrid head + feature MLP (training from scratch)
WARMUP_RATIO        = 0.10
WEIGHT_DECAY        = 0.01
LABEL_SMOOTHING     = 0.05
DROPOUT_BACKBONE    = 0.2
FREEZE_EPOCHS       = 3      # freeze backbone for first N epochs

# Valid image extensions
VALID_EXT = {'.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tiff'}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config OK')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
# ============================================================
# PHASE-1-SELECTED FEATURES
# Only the 7 features that had True AUC >= 0.61 in Phase 1
# Inverted features: score was flipped (high = more AI after inversion)
# ============================================================

FEATURE_NAMES = [
    'lbp_entropy',            # AUC 0.79  — best single signal
    'dct_blocking',           # AUC 0.69  — Benford's law on DCT
    'gradient_cooccurrence',  # AUC 0.68  — gradient texture richness
    'wavelet',                # AUC 0.66  — Haar diagonal energy
    'fft_slope',              # AUC 0.69* — power spectrum (inverted)
    'bayer_noise',            # AUC 0.62* — demosaic fingerprint (inverted)
    'edge_sharpness',         # AUC 0.61* — PSF spatial consistency (inverted)
]
NUM_FEATURES = len(FEATURE_NAMES)  # 7

# True = flip (1 - score) so that HIGH value always = more likely AI
FEATURE_INVERT = {
    'lbp_entropy':           False,
    'dct_blocking':          False,
    'gradient_cooccurrence': False,
    'wavelet':               False,
    'fft_slope':             True,
    'bayer_noise':           True,
    'edge_sharpness':        True,
}

print(f'Using {NUM_FEATURES} Phase-1 features')
for f in FEATURE_NAMES:
    inv = ' (inverted)' if FEATURE_INVERT[f] else ''
    print(f'  {f}{inv}')

In [ ]:
# ============================================================
# FORENSIC PREPROCESSING CLASSES
# Exact copies from Phase 1 old training code for consistency
# ============================================================

class AdaptiveResize:
    """
    Smart resize that preserves quality when downsampling large images.
    Used for val/test — NO augmentation.
    Step 1 (if > 2x target): LANCZOS to 2x target  (anti-aliasing)
    Step 2: BICUBIC to final target size
    NOTE: Phase 1 feature extraction used plain BICUBIC (no step-down).
          Features in features.csv were computed that way, so this class
          is ONLY used for the image tensor pipeline (DL branch), not
          for on-the-fly feature extraction.
    """
    def __init__(self, size=512):
        self.size = size

    def __call__(self, img):
        w, h = img.size
        if abs(w - self.size) < 50 and abs(h - self.size) < 50:
            return img.resize((self.size, self.size), Image.BICUBIC)
        max_dim = max(w, h)
        if max_dim > self.size * 2:
            scale = (self.size * 2) / max_dim
            new_w, new_h = int(w * scale), int(h * scale)
            img = img.resize((new_w, new_h), Image.LANCZOS)
        return img.resize((self.size, self.size), Image.BICUBIC)


class RandomJPEGCompression:
    """Simulate real-world JPEG compression during training."""
    def __init__(self, quality_range=(70, 98), p=0.4):
        self.quality_range = quality_range
        self.p = p

    def __call__(self, img):
        if random.random() < self.p:
            quality = random.randint(*self.quality_range)
            buf = BytesIO()
            if img.mode != 'RGB': img = img.convert('RGB')
            img.save(buf, format='JPEG', quality=quality)
            buf.seek(0)
            img = Image.open(buf).copy()
            buf.close()
        return img


class QualityBalancer:
    """Subtle Gaussian noise to prevent quality-based classification."""
    def __init__(self, p=0.25):
        self.p = p

    def __call__(self, img):
        if random.random() < self.p:
            arr = np.array(img)
            noise = np.random.normal(0, 1.5, arr.shape)
            img = Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))
        return img


print('Preprocessing classes ready')

In [ ]:
# ============================================================
# LOAD DATASET & BUILD TRAIN / VAL / TEST SPLITS
# Stratified: 70% train | 15% val | 15% test
# ============================================================

print('Loading image paths...')
image_paths, labels, sources = [], [], []
classes   = ['ai', 'hum']
class_map = {'ai': 0, 'hum': 1}

def load_images_recursive(folder, label):
    count = 0
    for root, _, files in os.walk(folder):
        sub = os.path.basename(root)
        for f in files:
            if os.path.splitext(f)[1].lower() in VALID_EXT:
                image_paths.append(os.path.join(root, f))
                labels.append(label)
                sources.append(f'{classes[label]}_{sub}')
                count += 1
    return count

ai_count   = load_images_recursive(AI_FOLDER,   class_map['ai'])
real_count = load_images_recursive(REAL_FOLDER, class_map['hum'])
total      = ai_count + real_count
print(f'AI: {ai_count:,}  Real: {real_count:,}  Total: {total:,}')

# Class weights for weighted loss
w_ai   = total / (2 * ai_count)
w_real = total / (2 * real_count)
class_weights_tensor = torch.tensor([w_ai, w_real]).to(DEVICE)
print(f'Class weights: AI={w_ai:.3f}  Real={w_real:.3f}')

# ---------- Stratified 3-way split ----------
# First: carve off test (15%)
idx = list(range(total))
idx_trainval, idx_test = train_test_split(
    idx, test_size=TEST_SIZE, stratify=labels, random_state=42)

# Then: split remaining into train / val
labels_trainval = [labels[i] for i in idx_trainval]
val_fraction    = VAL_SIZE / (1.0 - TEST_SIZE)  # e.g. 0.15/0.85 ≈ 0.176
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=val_fraction, stratify=labels_trainval, random_state=42)

print(f'Train: {len(idx_train):,} | Val: {len(idx_val):,} | Test: {len(idx_test):,}')
print(f'Train AI/Real: {sum(labels[i]==0 for i in idx_train)}/{sum(labels[i]==1 for i in idx_train)}')
print(f'Val   AI/Real: {sum(labels[i]==0 for i in idx_val)}/{sum(labels[i]==1 for i in idx_val)}')
print(f'Test  AI/Real: {sum(labels[i]==0 for i in idx_test)}/{sum(labels[i]==1 for i in idx_test)}')

In [ ]:
# ============================================================
# LOAD features.csv FROM PHASE 1
# Columns: path, label, lbp_entropy, dct_blocking, ...
# Apply inversion so high = more AI for all 7 features
# ============================================================

print(f'Loading Phase-1 features from {FEATURES_CSV}...')
feat_df = pd.read_csv(FEATURES_CSV)
print(f'  Loaded {len(feat_df)} rows, columns: {list(feat_df.columns)}')

# Verify all 7 feature columns exist
missing_cols = [f for f in FEATURE_NAMES if f not in feat_df.columns]
if missing_cols:
    raise ValueError(f'Missing columns in features.csv: {missing_cols}')

# Build path → feature-vector lookup
# Phase 1 saved features at 512x512 plain BICUBIC — consistent with our pipeline
feat_raw = {}
for _, row in feat_df.iterrows():
    vec = np.array([float(row[f]) for f in FEATURE_NAMES], dtype=np.float32)
    # Apply inversion: flip scores for features where Phase 1 showed inverted AUC
    for i, feat in enumerate(FEATURE_NAMES):
        if FEATURE_INVERT[feat]:
            vec[i] = 1.0 - vec[i]
    feat_raw[str(row['path'])] = vec

# Fill NaNs with 0.5 (neutral) and clip to [0,1]
for k in feat_raw:
    v = feat_raw[k]
    v = np.where(np.isnan(v), 0.5, v)
    feat_raw[k] = np.clip(v, 0.0, 1.0)

# Fit StandardScaler on ALL feature vectors (before split — features are deterministic)
all_vecs = np.stack(list(feat_raw.values()))
scaler = StandardScaler().fit(all_vecs)

feat_lookup = {k: scaler.transform(v.reshape(1,-1))[0].astype(np.float32)
               for k, v in feat_raw.items()}

print(f'Feature vectors ready: {all_vecs.shape}')
print(f'  mean={all_vecs.mean():.3f}  std={all_vecs.std():.3f}')

# Check coverage
covered = sum(1 for p in image_paths if str(p) in feat_lookup)
missing = total - covered
print(f'Feature coverage: {covered}/{total} images matched')
if missing > 0:
    print(f'  ⚠️  {missing} images not in features.csv — will use zero vector (neutral)')

In [ ]:
# ============================================================
# HF DATASET + TRANSFORMS
# ============================================================

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
normalize = Normalize(mean=processor.image_mean, std=processor.image_std)
ZERO_FEAT = np.zeros(NUM_FEATURES, dtype=np.float32)  # fallback for missing paths

# Training transforms — forensic-safe (NO ColorJitter, NO GaussianBlur)
train_transforms = Compose([
    RandomJPEGCompression(quality_range=(70, 98), p=0.4),
    QualityBalancer(p=0.25),
    RandomResizedCrop(
        INPUT_RESOLUTION,
        scale=(0.3, 1.0),
        ratio=(0.95, 1.05),
        interpolation=Image.BICUBIC,
        antialias=True,
    ),
    RandomHorizontalFlip(p=0.5),
    RandomRotation(degrees=3),
    ToTensor(),
    normalize,
])

# Val/test transforms — AdaptiveResize (step-down LANCZOS + BICUBIC, from old code)
val_transforms = Compose([
    AdaptiveResize(INPUT_RESOLUTION),
    ToTensor(),
    normalize,
])


def make_transform_fn(transforms, feat_lookup):
    def fn(examples):
        examples['pixel_values'] = [
            transforms(img.convert('RGB')) for img in examples['image']
        ]
        # Attach traditional features as a list of tensors
        examples['trad_features'] = [
            feat_lookup.get(str(p), ZERO_FEAT).tolist()
            for p in examples['path']
        ]
        del examples['image']
        if 'source' in examples: del examples['source']
        return examples
    return fn


def build_hf_dataset(idx_list):
    return Dataset.from_dict({
        'image':  [image_paths[i] for i in idx_list],
        'label':  [labels[i]      for i in idx_list],
        'source': [sources[i]     for i in idx_list],
        'path':   [image_paths[i] for i in idx_list],
    }).cast_column('image', HFImage()).cast(
        {**{'label': ClassLabel(names=classes)},
         **{k: v for k, v in
            Dataset.from_dict({'image':[], 'label':[], 'source':[], 'path':[]}).features.items()
            if k not in ('image','label')}}
    )


# Simpler approach — build directly
def make_split(idx_list):
    ds = Dataset.from_dict({
        'image':  [image_paths[i] for i in idx_list],
        'label':  [labels[i]      for i in idx_list],
        'path':   [image_paths[i] for i in idx_list],
    })
    ds = ds.cast_column('image', HFImage())
    feats = ds.features.copy()
    feats['label'] = ClassLabel(names=classes)
    return ds.cast(feats)


train_ds = make_split(idx_train)
val_ds   = make_split(idx_val)
test_ds  = make_split(idx_test)

train_ds = train_ds.with_transform(make_transform_fn(train_transforms, feat_lookup))
val_ds   = val_ds.with_transform(make_transform_fn(val_transforms,   feat_lookup))
test_ds  = test_ds.with_transform(make_transform_fn(val_transforms,  feat_lookup))

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

In [ ]:
# ============================================================
# COLLATE FUNCTION
# Stacks pixel_values + trad_features + labels into tensors
# ============================================================

def collate_fn(examples):
    pixel_values, trad_feats, lbls = [], [], []
    for ex in examples:
        pv  = ex.get('pixel_values')
        tf  = ex.get('trad_features')
        lbl = ex.get('label')
        if isinstance(pv, torch.Tensor) and tf is not None and lbl is not None:
            pixel_values.append(pv)
            trad_feats.append(torch.tensor(tf, dtype=torch.float32))
            lbls.append(lbl)
    if not pixel_values:
        return {'pixel_values': torch.empty(0),
                'trad_features': torch.empty(0),
                'labels': torch.empty(0, dtype=torch.long)}
    return {
        'pixel_values':  torch.stack(pixel_values),
        'trad_features': torch.stack(trad_feats),
        'labels':        torch.tensor(lbls, dtype=torch.long),
    }

print('Collate function ready')

In [ ]:
# ============================================================
# MODEL ARCHITECTURE
# ============================================================

# ── Resize position embeddings (from old training code, exact copy) ──────────
def resize_siglip_embeddings(model, new_resolution):
    """
    Bicubic-interpolates SigLIP position embeddings from 224-pretrained
    grid to 512x512 grid.
    224px / 16px patch = 14x14 = 196 patches → 512px / 16px = 32x32 = 1024 patches
    """
    print(f'Resizing position embeddings to {new_resolution}x{new_resolution}...')
    vision_model = None
    if hasattr(model, 'vision_model'):         vision_model = model.vision_model
    elif hasattr(model, 'siglip'):             vision_model = model.siglip.vision_model
    elif hasattr(model, 'vit'):                vision_model = model.vit
    if vision_model is None:
        print('  ⚠️  Could not locate vision backbone, skipping'); return

    embeddings   = vision_model.embeddings
    patch_size   = embeddings.patch_size
    new_n_patches = (new_resolution // patch_size) ** 2
    old_n_patches = embeddings.num_patches

    if old_n_patches == new_n_patches:
        print('  ✅ Already correct size'); return

    print(f'  {old_n_patches} → {new_n_patches} patches  '
          f'(patch_size={patch_size})')

    old_pos = embeddings.position_embedding.weight.data
    dim     = old_pos.shape[-1]
    old_g   = int(old_n_patches ** 0.5)
    new_g   = int(new_n_patches ** 0.5)

    # (N, D) → (1, D, g, g) → bicubic → (1, D, G, G) → (N', D)
    old_pos = old_pos.reshape(1, old_g, old_g, dim).permute(0, 3, 1, 2)
    new_pos = F.interpolate(old_pos, size=(new_g, new_g),
                            mode='bicubic', align_corners=False)
    new_pos = new_pos.permute(0, 2, 3, 1).reshape(new_n_patches, dim)

    new_emb = nn.Embedding(new_n_patches, dim)
    new_emb.weight.data = new_pos.to(next(model.parameters()).device)
    embeddings.position_embedding = new_emb
    embeddings.num_patches  = new_n_patches
    embeddings.image_size   = new_resolution
    embeddings.register_buffer('position_ids',
                               torch.arange(new_n_patches).expand((1, -1)))
    model.config.vision_config.image_size = new_resolution
    print(f'  ✅ Position embeddings resized to {new_resolution}x{new_resolution}')


# ── SE-style channel attention (same as ArtifactAttention in old code) ────────
class ArtifactAttention(nn.Module):
    """
    Squeeze-and-Excitation style channel attention.
    Re-calibrates feature importance — exact architecture from old code.
    squeeze ratio = 4  (hidden_dim → hidden_dim//4 → hidden_dim)
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, hidden_dim),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.attention(x)  # channel-wise gating


# ── Small MLP to embed 7 traditional features into 128-d ─────────────────────
class FeatureMLP(nn.Module):
    def __init__(self, in_d=7, hidden=64, out_d=128, drop=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_d, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden, hidden * 2),
            nn.LayerNorm(hidden * 2),
            nn.GELU(),
            nn.Dropout(drop * 0.8),
            nn.Linear(hidden * 2, out_d),
        )
    def forward(self, x): return self.net(x)


# ── Hybrid forensic head: 3-layer with SE-attention (extends old ForensicClassifier)
class HybridForensicHead(nn.Module):
    """
    3-layer classification head with SE-style ArtifactAttention.
    Takes concatenated [backbone_emb | feat_emb] as input.

    Layer 1: in_features → 1024, BN, GELU, Dropout(0.4)
    SE gate:  ArtifactAttention(1024)  ← same as old code
    Layer 2: 1024 → 512,  BN, GELU, Dropout(0.3)
    Layer 3: 512  → num_classes
    """
    def __init__(self, in_features, num_classes=2):
        super().__init__()
        self.fc1       = nn.Linear(in_features, 1024)
        self.bn1       = nn.BatchNorm1d(1024)
        self.drop1     = nn.Dropout(0.4)
        self.attention = ArtifactAttention(1024)   # SE gate
        self.fc2       = nn.Linear(1024, 512)
        self.bn2       = nn.BatchNorm1d(512)
        self.drop2     = nn.Dropout(0.3)
        self.fc3       = nn.Linear(512, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.drop1(F.gelu(self.bn1(self.fc1(x))))
        x = self.attention(x)                        # SE channel re-weighting
        x = self.drop2(F.gelu(self.bn2(self.fc2(x))))
        return self.fc3(x)


# ── Full hybrid model ─────────────────────────────────────────────────────────
class HybridAIDetector(nn.Module):
    def __init__(self, base_model, backbone_dim, num_features=7,
                 feat_embed_dim=128, num_classes=2):
        super().__init__()
        self.backbone   = base_model
        self.feat_mlp   = FeatureMLP(in_d=num_features, out_d=feat_embed_dim)
        fused_dim       = backbone_dim + feat_embed_dim
        self.head       = HybridForensicHead(fused_dim, num_classes)

    def freeze_backbone(self, freeze=True):
        for p in self.backbone.parameters():
            p.requires_grad = not freeze

    def forward(self, pixel_values, trad_features, labels=None):
        # Backbone forward — get pooled embedding
        out = self.backbone(pixel_values=pixel_values,
                            output_hidden_states=True)
        # Use pooler if available, else mean-pool last hidden state
        if hasattr(out, 'pooler_output') and out.pooler_output is not None:
            img_emb = out.pooler_output
        else:
            img_emb = out.hidden_states[-1].mean(dim=1)

        # Feature branch
        feat_emb = self.feat_mlp(trad_features)

        # Fuse and classify
        fused  = torch.cat([img_emb, feat_emb], dim=-1)
        logits = self.head(fused)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        from transformers.modeling_outputs import ImageClassifierOutput
        return ImageClassifierOutput(loss=loss, logits=logits)

print('Model classes defined')

In [ ]:
# ============================================================
# INSTANTIATE MODEL
# ============================================================

print(f'Loading {MODEL_ID}...')

# Load the fine-tuned backbone (Ateeqq model = SigLIP fine-tuned on AI vs Human)
# We load with AutoModelForImageClassification to get the full HF model,
# then extract the backbone and attach our hybrid head
base = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    ignore_mismatched_sizes=True,
    output_hidden_states=True,
)

# Resize position embeddings: 224 → 512  (bicubic interpolation of 14x14→32x32 grid)
resize_siglip_embeddings(base, INPUT_RESOLUTION)

# Get backbone dimension from existing classifier
if isinstance(base.classifier, nn.Linear):
    backbone_dim = base.classifier.in_features
elif isinstance(base.classifier, nn.Sequential):
    for layer in base.classifier:
        if isinstance(layer, nn.Linear):
            backbone_dim = layer.in_features; break
else:
    backbone_dim = base.config.hidden_size

print(f'Backbone embedding dim: {backbone_dim}')

# Remove the old classification head — we attach HybridForensicHead instead
base.classifier = nn.Identity()

# Build hybrid model
model = HybridAIDetector(
    base_model     = base,
    backbone_dim   = backbone_dim,
    num_features   = NUM_FEATURES,
    feat_embed_dim = 128,
    num_classes    = 2,
)
model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params/1e6:.1f}M')
print(f'Trainable params: {trainable_params/1e6:.1f}M')
print(f'Fused input dim:  {backbone_dim + 128}  ({backbone_dim} backbone + 128 feature MLP)')

In [ ]:
# ============================================================
# WEIGHTED TRAINER WITH LABEL SMOOTHING
# Exact same pattern as old WeightedTrainer
# ============================================================

class HybridWeightedTrainer(Trainer):
    """
    Custom Trainer that:
    1. Passes trad_features to the model forward
    2. Uses class-weighted cross-entropy + label smoothing
    3. Supports backbone freeze/unfreeze schedule via epoch callbacks
    """
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights  = class_weights
        self._current_epoch = 0

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels       = inputs.pop('labels')
        trad_features = inputs.pop('trad_features')
        pixel_values = inputs['pixel_values']

        outputs = model(pixel_values=pixel_values,
                        trad_features=trad_features,
                        labels=labels)
        logits  = outputs.logits

        loss_fn = nn.CrossEntropyLoss(
            weight=self.class_weights,
            label_smoothing=LABEL_SMOOTHING,
        )
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        """Override to pass trad_features during eval."""
        labels       = inputs.get('labels')
        trad_features = inputs.get('trad_features')
        pixel_values = inputs.get('pixel_values')

        with torch.no_grad():
            outputs = model(pixel_values=pixel_values.to(DEVICE),
                            trad_features=trad_features.to(DEVICE),
                            labels=labels.to(DEVICE) if labels is not None else None)
        loss   = outputs.loss
        logits = outputs.logits

        if prediction_loss_only:
            return (loss, None, None)
        return (loss, logits, labels)


print('HybridWeightedTrainer defined')

In [ ]:
# ============================================================
# METRICS — same detailed breakdown as old code
# ============================================================

def compute_metrics(eval_pred) -> Dict[str, float]:
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    # AUC
    probs = predictions[:, 1] if predictions.ndim == 2 else predictions
    try:   auc = roc_auc_score(labels, probs)
    except: auc = 0.0

    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        ai_acc   = tn / (tn + fp) if (tn + fp) > 0 else 0
        real_acc = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr      = fp / (fp + tn) if (fp + tn) > 0 else 0
        fnr      = fn / (fn + tp) if (fn + tp) > 0 else 0
    else:
        ai_acc = real_acc = fpr = fnr = 0.0

    return {
        'accuracy':     acc,
        'f1':           f1,
        'precision':    precision,
        'recall':       recall,
        'auc':          auc,
        'ai_accuracy':  ai_acc,
        'real_accuracy':real_acc,
        'fpr':          fpr,
        'fnr':          fnr,
    }


print('Metrics defined')

In [ ]:
# ============================================================
# BACKBONE FREEZE CALLBACK
# Freeze backbone for FREEZE_EPOCHS, then unfreeze for joint fine-tuning
# ============================================================

from transformers import TrainerCallback

class FreezeUnfreezeCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, **kwargs):
        epoch = int(state.epoch) if state.epoch else 0
        mm = kwargs['model']
        if epoch == 0:
            mm.freeze_backbone(True)
            print(f'  [Epoch {epoch+1}] Backbone FROZEN — training feature MLP + head only')
        elif epoch == FREEZE_EPOCHS:
            mm.freeze_backbone(False)
            print(f'  [Epoch {epoch+1}] Backbone UNFROZEN — full fine-tuning')

print('FreezeUnfreezeCallback defined')

In [ ]:
# ============================================================
# TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = MAX_EPOCHS,

    # Batch / gradient accumulation
    per_device_train_batch_size = BATCH_SIZE_PER_GPU,
    per_device_eval_batch_size  = BATCH_SIZE_PER_GPU,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,

    # Optimization
    learning_rate               = LEARNING_RATE,   # backbone LR
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    lr_scheduler_type           = 'cosine',

    # Evaluation & saving
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,

    # Logging
    logging_dir                 = f'{OUTPUT_DIR}/logs',
    logging_steps               = 25,

    # Performance
    fp16                        = True,
    dataloader_num_workers      = 4,
    dataloader_pin_memory       = True,
    gradient_checkpointing      = False,

    remove_unused_columns       = False,
    push_to_hub                 = False,
    report_to                   = ['tensorboard'],
)

print('TrainingArguments ready')
eff_bs = BATCH_SIZE_PER_GPU * max(NUM_GPUS, 1) * GRAD_ACCUM_STEPS
print(f'Effective batch size: {eff_bs}')

In [ ]:
# ============================================================
# TRAINER
# ============================================================

trainer = HybridWeightedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    data_collator   = collate_fn,
    compute_metrics = compute_metrics,
    class_weights   = class_weights_tensor,
    callbacks       = [
        FreezeUnfreezeCallback(),
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE),
    ],
)

print('Trainer ready')

In [ ]:
# ============================================================
# TRAIN
# ============================================================

print('\n' + '='*65)
print('STARTING HYBRID TRAINING @ 512x512')
print(f'  Model:      {MODEL_ID}')
print(f'  Features:   {NUM_FEATURES} Phase-1 signals')
print(f'  Head:       3-layer + SE-ArtifactAttention (1024→SE→512→2)')
print(f'  Freeze:     backbone frozen for first {FREEZE_EPOCHS} epochs')
print(f'  Split:      Train={len(train_ds)} Val={len(val_ds)} Test={len(test_ds)}')
print('='*65 + '\n')

train_result = trainer.train()
trainer.log_metrics('train', train_result.metrics)
trainer.save_metrics('train', train_result.metrics)

print('\n✅ TRAINING COMPLETE')

In [ ]:
# ============================================================
# EVALUATE ON VALIDATION SET
# ============================================================

print('\nEvaluating on VAL set...')
val_metrics = trainer.evaluate(eval_dataset=val_ds)
trainer.log_metrics('eval', val_metrics)
trainer.save_metrics('eval', val_metrics)

print('\n' + '='*50)
print('VAL RESULTS')
print('='*50)
for k, v in val_metrics.items():
    if isinstance(v, float): print(f'  {k:20s}: {v:.4f}')
print('='*50)

In [ ]:
# ============================================================
# EVALUATE ON HELD-OUT TEST SET
# ============================================================

print('\nEvaluating on TEST set (held-out)...')
test_output = trainer.predict(test_ds)

y_true  = test_output.label_ids
y_pred  = test_output.predictions.argmax(1)
y_probs = test_output.predictions[:, 1]

from sklearn.metrics import roc_auc_score
test_acc  = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
test_auc  = roc_auc_score(y_true, y_probs)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print('\n' + '='*50)
print('FINAL TEST RESULTS')
print('='*50)
print(f'  Accuracy   : {test_acc:.4f}')
print(f'  F1         : {f1:.4f}')
print(f'  Precision  : {prec:.4f}')
print(f'  Recall     : {rec:.4f}')
print(f'  AUC-ROC    : {test_auc:.4f}')
print(f'  AI acc.    : {tn/(tn+fp):.4f}  (TN rate)')
print(f'  Real acc.  : {tp/(tp+fn):.4f}  (TP rate)')
print(f'  FPR        : {fp/(fp+tn):.4f}')
print(f'  FNR        : {fn/(fn+tp):.4f}')
print('='*50)
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=classes))

In [ ]:
# ============================================================
# CONFUSION MATRIX PLOT
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
ax = axes[0]
im = ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=16, fontweight='bold',
                color='white' if cm[i, j] > cm.max() * 0.5 else 'black')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['AI (pred)', 'Real (pred)'])
ax.set_yticklabels(['AI (true)', 'Real (true)'])
ax.set_title('Confusion Matrix — Test Set', fontweight='bold')
plt.colorbar(im, ax=ax)

# Metric summary bar
ax2 = axes[1]
metric_names  = ['Accuracy', 'F1', 'Precision', 'Recall', 'AUC-ROC']
metric_values = [test_acc, f1, prec, rec, test_auc]
colors = ['#22c55e' if v >= 0.90 else ('#f59e0b' if v >= 0.80 else '#ef4444')
          for v in metric_values]
bars = ax2.barh(metric_names, metric_values, color=colors, edgecolor='white')
for bar, v in zip(bars, metric_values):
    ax2.text(v + 0.005, bar.get_y() + bar.get_height() / 2,
             f'{v:.4f}', va='center', fontsize=11, fontweight='bold')
ax2.set_xlim(0, 1.08)
ax2.axvline(0.9, color='green', linestyle=':', alpha=0.5, label='0.90 target')
ax2.set_title('Test Metrics Summary', fontweight='bold')
ax2.legend()

plt.suptitle('Hybrid AI Detector — Final Test Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/test_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved test_results.png')

In [ ]:
# ============================================================
# SAVE — HF-compatible backbone + hybrid head weights
# ============================================================

# Save backbone as a standard HF model (compatible with app.py)
backbone_out = f'{OUTPUT_DIR}/backbone'
model.backbone.save_pretrained(backbone_out)
processor.save_pretrained(backbone_out)
print(f'Backbone saved → {backbone_out}')

# Save hybrid head + feature MLP + scaler config
torch.save({
    'feat_mlp':      model.feat_mlp.state_dict(),
    'head':          model.head.state_dict(),
    'scaler_mean':   scaler.mean_,
    'scaler_std':    scaler.scale_,
    'feature_names': FEATURE_NAMES,
    'feature_invert': FEATURE_INVERT,
    'backbone_dim':  backbone_dim,
    'feat_embed_dim': 128,
    'num_classes':   2,
    'input_resolution': INPUT_RESOLUTION,
}, f'{OUTPUT_DIR}/hybrid_head.pt')
print(f'Hybrid head → {OUTPUT_DIR}/hybrid_head.pt')

# Save config JSON
with open(f'{OUTPUT_DIR}/hybrid_config.json', 'w') as f:
    json.dump({
        'model_id':          MODEL_ID,
        'input_resolution':  INPUT_RESOLUTION,
        'feature_names':     FEATURE_NAMES,
        'feature_invert':    FEATURE_INVERT,
        'num_features':      NUM_FEATURES,
        'test_accuracy':     float(test_acc),
        'test_f1':           float(f1),
        'test_auc':          float(test_auc),
    }, f, indent=2)

# Zip everything for download
import shutil
shutil.make_archive('/kaggle/working/hybrid_forensic_512', 'zip', OUTPUT_DIR)
print('\n hybrid_forensic_512.zip ready for download')
print('Contents:')
print('  backbone/          ← HF-compatible model (for app.py)')
print('  hybrid_head.pt     ← feature MLP + 3-layer SE head weights')
print('  hybrid_config.json ← feature names, inversion flags, scaler')

In [ ]:
from IPython.display import FileLink
FileLink(f"{"hybrid_forensic_512"}.zip")